# ML-09 — Validation and Research Claim Audit

This notebook practices the same rigor on my own model that the research paper asks from its readers: identify the claim, ask where the label comes from, test whether the validation design supports the claim, and then attack my own features for leakage before I trust the result.

## 1. Two paper findings + my methodology questions

Finding 1: the paper's refresh model reports strong ranking performance, with random forest selected by `precision@50`. My methodology question is simple: where does the label come from, and is the label definition separated cleanly enough from the input features that the ranking is measuring real generalization instead of a window overlap or label-derived shortcut?

Finding 2: the paper says its main validation style is client-holdout. My methodology question is whether that split really supports the claim being made. If the claim is that the method works on unseen clients, then keeping whole clients out is the right test; if the claim were broader, I would want to know whether the same behavior survives a different validation design too.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, average_precision_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split

repo_root = next(
    (candidate for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent] if (candidate / "scripts" / "ml_utils.py").exists()),
    None,
 )
if repo_root is None:
    raise FileNotFoundError("Could not locate the repository root.")

sys.path.insert(0, str(repo_root / "scripts"))
from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES, precision_at_k

feature_path = repo_root / "data" / "processed" / "refresh_feature_vector.csv"
baseline_path = repo_root / "data" / "processed" / "baseline_refresh_queue.csv"

frame = pd.read_csv(feature_path)
baseline_frame = pd.read_csv(baseline_path)
baseline_lookup = baseline_frame.set_index("content_id")["baseline_refresh_score"]

RANDOM_STATE = 42

def build_feature_matrix(frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    numeric_features = [column for column in MODEL_NUMERIC_FEATURES if column in frame.columns]
    categorical_features = [column for column in MODEL_CATEGORICAL_FEATURES if column in frame.columns]

    numeric_frame = frame[numeric_features].apply(pd.to_numeric, errors="coerce")
    numeric_frame = numeric_frame.replace([np.inf, -np.inf], np.nan).fillna(0)

    categorical_frame = frame[categorical_features].fillna("unknown").astype(str)
    encoded_frame = pd.get_dummies(
        categorical_frame,
        prefix=categorical_features,
        dummy_na=False,
        dtype=float,
    )

    feature_frame = pd.concat(
        [numeric_frame.reset_index(drop=True), encoded_frame.reset_index(drop=True)],
        axis=1,
    )
    return feature_frame, list(feature_frame.columns)

def metric_payload(y_true: pd.Series, scores: np.ndarray) -> dict[str, float]:
    binary_predictions = (scores >= 0.5).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, binary_predictions)),
        "precision": float(precision_score(y_true, binary_predictions, zero_division=0)),
        "recall": float(recall_score(y_true, binary_predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, scores)) if y_true.nunique() == 2 else 0.0,
        "average_precision": float(average_precision_score(y_true, scores)) if y_true.nunique() == 2 else 0.0,
        "precision_at_20": float(precision_at_k(y_true, scores, 20)),
        "precision_at_50": float(precision_at_k(y_true, scores, 50)),
        "precision_at_100": float(precision_at_k(y_true, scores, 100)),
    }

def train_rf_on_split(frame: pd.DataFrame, split_kind: str) -> dict[str, object]:
    feature_frame, feature_columns = build_feature_matrix(frame)
    target = frame["is_declining_label"].astype(int)
    groups = frame["client_id"].astype(str)

    if split_kind == "client_holdout":
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
        train_idx, test_idx = next(splitter.split(feature_frame, target, groups=groups))
        split_strategy = "client_holdout"
    else:
        train_idx, test_idx = train_test_split(
            np.arange(len(frame)),
            test_size=0.2,
            random_state=RANDOM_STATE,
            stratify=target,
        )
        train_idx = np.asarray(train_idx)
        test_idx = np.asarray(test_idx)
        split_strategy = "row_holdout"

    model = RandomForestClassifier(
        class_weight="balanced_subsample",
        max_depth=10,
        min_samples_leaf=25,
        n_estimators=200,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    model.fit(feature_frame.iloc[train_idx], target.iloc[train_idx])
    probabilities = model.predict_proba(feature_frame.iloc[test_idx])[:, 1]

    baseline_scores = frame.iloc[test_idx]["content_id"].map(baseline_lookup).fillna(0).to_numpy()

    test_frame = frame.iloc[test_idx][[
        "content_id",
        "client_id",
        "content_type",
        "is_declining_label",
        "impressions_90d",
        "sessions_90d",
        "ctr",
        "avg_position",
        "content_age_days",
        "days_since_last_update",
    ]].copy()
    test_frame["baseline_refresh_score"] = baseline_scores
    test_frame["model_probability"] = probabilities
    test_frame["predicted_label"] = (test_frame["model_probability"] >= 0.5).astype(int)

    error_frame = test_frame.assign(error=(test_frame["predicted_label"] != test_frame["is_declining_label"]))

    return {
        "split_strategy": split_strategy,
        "train_rows": int(len(train_idx)),
        "test_rows": int(len(test_idx)),
        "base_rate": float(target.iloc[test_idx].mean()),
        "baseline_metrics": metric_payload(target.iloc[test_idx], baseline_scores),
        "model_metrics": metric_payload(target.iloc[test_idx], probabilities),
        "test_frame": test_frame,
        "error_frame": error_frame,
        "feature_columns": feature_columns,
        "model": model,
    }

frame["client_id"] = frame["client_id"].astype(str)
honest_result = train_rf_on_split(frame, "client_holdout")
row_result = train_rf_on_split(frame, "row_holdout")

before_after = pd.DataFrame(
    [
        {
            "split": row_result["split_strategy"],
            "baseline_precision_at_50": row_result["baseline_metrics"]["precision_at_50"],
            "model_precision_at_50": row_result["model_metrics"]["precision_at_50"],
            "model_roc_auc": row_result["model_metrics"]["roc_auc"],
            "base_rate": row_result["base_rate"],
        },
        {
            "split": honest_result["split_strategy"],
            "baseline_precision_at_50": honest_result["baseline_metrics"]["precision_at_50"],
            "model_precision_at_50": honest_result["model_metrics"]["precision_at_50"],
            "model_roc_auc": honest_result["model_metrics"]["roc_auc"],
            "base_rate": honest_result["base_rate"],
        },
    ]
)

before_after["model_minus_baseline_precision_at_50"] = (
    before_after["model_precision_at_50"] - before_after["baseline_precision_at_50"]
 )

display(before_after)

print(
    f"Row-holdout model Precision@50: {row_result['model_metrics']['precision_at_50']:.3f}; "
    f"client-holdout model Precision@50: {honest_result['model_metrics']['precision_at_50']:.3f}; "
    f"gap: {row_result['model_metrics']['precision_at_50'] - honest_result['model_metrics']['precision_at_50']:.3f}"
 )

honest_top_errors = honest_result["error_frame"].sort_values(["error", "model_probability"], ascending=[False, False]).head(10)
display(honest_top_errors[[
    "content_id",
    "client_id",
    "content_type",
    "is_declining_label",
    "model_probability",
    "predicted_label",
    "baseline_refresh_score",
    "error",
    "impressions_90d",
    "avg_position",
    "days_since_last_update",
]])

,split,baseline_precision_at_50,model_precision_at_50,model_roc_auc,base_rate,model_minus_baseline_precision_at_50
0,row_holdout,0.48,0.90,0.757888,0.542000,0.42
1,client_holdout,0.32,0.54,0.609623,0.510952,0.22


Row-holdout model Precision@50: 0.900; client-holdout model Precision@50: 0.540; gap: 0.360


,content_id,client_id,content_type,is_declining_label,model_probability,predicted_label,baseline_refresh_score,error,impressions_90d,avg_position,days_since_last_update
22042,content_2ba626fea4d6,client_8527a891e2,keyword article,0,0.858319,1,0.512588,True,360,7.2,104
10080,content_35d63627bf3e,client_8527a891e2,keyword article,0,0.851648,1,0.528319,True,1525,32.6,103
5011,content_c148e44db30d,client_8527a891e2,keyword article,0,0.846629,1,0.457740,True,335,31.3,104
26547,content_dbb4c75afccc,client_4e07408562,keyword article,0,0.846212,1,0.498727,True,636,33.9,104
22526,content_1d0963b56227,client_4e07408562,keyword article,0,0.846095,1,0.616138,True,3445,39.0,104
22524,content_846bb4dd8b44,client_8527a891e2,keyword article,0,0.845415,1,0.568305,True,870,17.6,104
11061,content_0b47dae0c7f9,client_8527a891e2,keyword article,0,0.842272,1,0.537547,True,1191,23.1,103
22461,content_7e3be2e230f5,client_4e07408562,keyword article,0,0.838829,1,0.530194,True,909,33.4,104
20736,content_41baf0722ad9,client_8527a891e2,keyword article,0,0.838642,1,0.705430,True,3115,12.8,104
12069,content_ff4370afd49c,client_4e07408562,keyword article,0,0.837012,1,0.579501,True,1677,33.1,104


## 2. My model under an honest split (before/after)

I compare the same random forest under a random row split and a client-holdout split. The grouped split is the honest version for this lane because pages from the same client share hidden structure; if the score drops, that drop is a feature of the validation, not a mistake in the math.

## 3. Leakage audit

The final feature set should only contain signals available before the decision moment. If a suspect column can reproduce the label too well, that is not model skill; it is a shortcut. I check the published feature set first, then I add one deliberate leak to confirm the harness catches it.

In [2]:
feature_frame, feature_columns = build_feature_matrix(frame)
target = frame["is_declining_label"].astype(int)

suspect_columns = ["is_declining_label", "trend_direction", "trend_pct", "content_id", "client_id"]
published_feature_set = [column for column in feature_columns if column not in suspect_columns]

leakage_audit = pd.DataFrame(
    [
        {
            "check": "Label-derived columns excluded",
            "result": "PASS" if "is_declining_label" not in feature_columns else "FAIL",
            "detail": "The final feature matrix does not use the target column.",
        },
        {
            "check": "Trend columns excluded",
            "result": "PASS" if ("trend_direction" not in feature_columns and "trend_pct" not in feature_columns) else "FAIL",
            "detail": "The label source columns stay out of the model inputs.",
        },
        {
            "check": "IDs used only for grouping and joins",
            "result": "PASS" if ("content_id" not in feature_columns and "client_id" not in feature_columns) else "FAIL",
            "detail": "IDs are context, not model features.",
        },
    ]
)

display(leakage_audit)

leaky_feature_frame = feature_frame.copy()
leaky_feature_frame["label_copy"] = target

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(feature_frame, target, groups=frame["client_id"].astype(str)))

honest_model = RandomForestClassifier(
    class_weight="balanced_subsample",
    max_depth=10,
    min_samples_leaf=25,
    n_estimators=200,
    n_jobs=-1,
    random_state=RANDOM_STATE,
 )
honest_model.fit(feature_frame.iloc[train_idx], target.iloc[train_idx])
honest_prob = honest_model.predict_proba(feature_frame.iloc[test_idx])[:, 1]

leaky_model = RandomForestClassifier(
    class_weight="balanced_subsample",
    max_depth=10,
    min_samples_leaf=25,
    n_estimators=200,
    n_jobs=-1,
    random_state=RANDOM_STATE,
 )
leaky_model.fit(leaky_feature_frame.iloc[train_idx], target.iloc[train_idx])
leaky_prob = leaky_model.predict_proba(leaky_feature_frame.iloc[test_idx])[:, 1]

leak_demo = pd.DataFrame(
    [
        {
            "setup": "honest features",
            **metric_payload(target.iloc[test_idx], honest_prob),
        },
        {
            "setup": "label copy added",
            **metric_payload(target.iloc[test_idx], leaky_prob),
        },
    ]
)

display(leak_demo[["setup", "roc_auc", "average_precision", "precision_at_50", "precision_at_20"]])

leaky_top_importance = pd.Series(leaky_model.feature_importances_, index=published_feature_set + ["label_copy"]).sort_values(ascending=False).head(10)
display(leaky_top_importance.to_frame(name="importance"))

print("The deliberate leak test should jump sharply if the harness is working; if it does, the honest feature set is the one to keep.")

,check,result,detail
0,Label-derived columns excluded,PASS,The final feature matrix does not use the targ...
1,Trend columns excluded,PASS,The label source columns stay out of the model...
2,IDs used only for grouping and joins,PASS,"IDs are context, not model features."


,setup,roc_auc,average_precision,precision_at_50,precision_at_20
0,honest features,0.609623,0.589574,0.54,0.5
1,label copy added,1.000000,1.000000,1.00,1.0


,importance
label_copy,0.783524
days_with_impressions,0.038092
log_impressions_90d,0.026830
avg_position,0.023788
content_age_days,0.019355
word_count,0.010529
char_count,0.010096
position_tier_top_3,0.009009
age_tier_365+,0.006244
ctr,0.005721


The deliberate leak test should jump sharply if the harness is working; if it does, the honest feature set is the one to keep.


## 4. Claim rewrite

My boldest draft claim would be: 'the model knows which pages to refresh first.' That is too strong. The evidence only supports a narrower claim about ranking on this anonymized starter slice under a client-holdout split.

In [3]:
safe_claim = pd.DataFrame(
    [
        {
            "claim_type": "unsafe draft",
            "statement": "The model knows which pages to refresh first.",
        },
        {
            "claim_type": "safe rewrite",
            "statement": "On this anonymized starter slice, the client-holdout random forest ranked more decline-labeled pages into the top 50 than the baseline rules, which is decision-support evidence rather than a causal claim.",
        },
        {
            "claim_type": "scope note",
            "statement": "The result is directional and measured on one held-out split; it should not be read as proof of recovery, causality, or deployment performance on the full warehouse.",
        },
    ]
)

display(safe_claim)

print(
    f"Base rate on honest test split: {honest_result['base_rate']:.3f}. "
    f"Client-holdout model Precision@50: {honest_result['model_metrics']['precision_at_50']:.3f}. "
    f"Baseline Precision@50: {honest_result['baseline_metrics']['precision_at_50']:.3f}."
 )

,claim_type,statement
0,unsafe draft,The model knows which pages to refresh first.
1,safe rewrite,"On this anonymized starter slice, the client-h..."
2,scope note,The result is directional and measured on one ...


Base rate on honest test split: 0.511. Client-holdout model Precision@50: 0.540. Baseline Precision@50: 0.320.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.